# Tool Calling Agents — 데스크탑 Ollama판

코스 섹션 [ToolCallingAgents](https://huggingface.co/learn/agents-course/unit2/smolagents/tool_calling_agents)의 예제를 **내 데스크탑 Ollama(`qwen3.5:9b`)**로 돌리고, 지난번 `CodeAgent`와 **나란히 비교**해 봐요.

| 단계 | 하는 일 | 확인할 것 |
|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | 지난 노트북(`code_agents_ollama.ipynb`)과 같음 |
| 2 | 원문 예제 — `ToolCallingAgent` + 웹 검색 | 로그에 `Executing parsed code:` 대신 **`Calling tool: 'web_search' with arguments: {...}`** |
| 3 | 속 들여다보기 — 모델이 돌려준 **도구 호출(JSON)** | 답이 글이 아니라 `tool_calls`라는 **별도 칸**으로 옴 |
| 4 | 비교 실험 — 같은 문제를 `CodeAgent` vs `ToolCallingAgent`로 | 바퀴 수, 비교를 누가 하나(코드의 `if` vs 모델의 머리) |

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 새로 설치할 것은 없어요.

## 0단계: 준비 확인

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "markdownify", "requests"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, 나머지 전부 `True`
- `ModuleNotFoundError` → venv를 켜지 않고 Jupyter를 열었음
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결

지난 노트북과 **같은 설정**이에요.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말.

## 2단계: 원문 예제 — `ToolCallingAgent` + 웹 검색

원문 코드에서 `model=InferenceClientModel()` → `model=model`만 바꿨어요. 지난번처럼 `max_steps=6`을 넣었어요.
- `WebSearchTool()` = 기본 제공 웹 검색 도구(DuckDuckGo). 지난번 `DuckDuckGoSearchTool()`과 이름(`web_search`)·하는 일이 같은 **형제 도구**
- `CodeAgent` → `ToolCallingAgent`로 **이름 한 단어만** 바뀜. 나머지는 프레임워크가 처리

In [ ]:
from smolagents import ToolCallingAgent, WebSearchTool

agent = ToolCallingAgent(tools=[WebSearchTool()], model=model, max_steps=6)

answer = agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- 상자 안에 코드가 아니라 **`Calling tool: 'web_search' with arguments: {'query': '...'}`** 가 보임 = 모델이 "어떤 도구를, 어떤 인자로"만 적어 보냄
- 그 아래 `Observations:` = 검색 결과 (지난번 `Execution logs:` 자리)
- 마지막에는 **`Calling tool: 'final_answer' with arguments: {'answer': '...'}`** — 최종 답도 도구 호출로 냄
- 막히면:
  - `RatelimitException`·`No results found` = 검색 사이트가 잠깐 막음 → 1~2분 뒤 다시. 계속 안 되면 첫 줄을 `from smolagents import ToolCallingAgent, DuckDuckGoSearchTool`로, `WebSearchTool()`을 `DuckDuckGoSearchTool()`로 바꿔서 실행 (지난번에 잘 된 도구)
  - `does not support tools` = 모델이 도구 호출 기능을 지원하지 않는다는 뜻 → 결과를 그대로 붙여 주세요

## 3단계: 속 들여다보기 — 모델이 돌려준 도구 호출

`CodeAgent` 때는 모델의 답이 **글(코드가 섞인 글)**이었고, smolagents가 그 글에서 코드를 꺼냈어요.
`ToolCallingAgent`는 모델 API에 "이 도구들이 있어"(`tools`)를 **따로** 알려 주고, 모델도 답을 **`tool_calls`라는 별도 칸**에 JSON으로 돌려줘요.

In [ ]:
from smolagents.memory import ActionStep

for step in agent.memory.steps:
    if not isinstance(step, ActionStep):
        continue
    msg = step.model_output_message
    print(f"===== {step.step_number}바퀴 =====")
    print("글(content):", repr((msg.content or "")[:200]))
    for tc in msg.tool_calls or []:
        print("도구 호출 → 이름:", tc.function.name, "| 인자:", tc.function.arguments)
    print()

✅ **확인할 것**
- `도구 호출 → 이름: web_search | 인자: {'query': ...}` = 원문의 JSON 덩어리 `{"name": ..., "arguments": ...}` 그 자체
- `글(content)`은 비어 있거나(`''`) 짧은 말뿐 → 도구 호출이 글 속에 섞여 오는 게 아님. **Unit 1 Dummy Agent처럼 글에서 JSON을 오려 낼 필요가 없음** (모델 API가 칸을 나눠 줌)
- 한 바퀴에 도구 호출이 **2개 이상** 보이면 = 모델이 한 번에 여러 도구를 부른 것 (smolagents가 동시에 실행)

## 4단계: 비교 실험 — CodeAgent vs ToolCallingAgent

가짜 도구 2개로 **같은 문제**를 두 에이전트에게 줘요. 답이 고정이라 인터넷이 필요 없고, 비교가 깔끔해요.
- `get_guest_count(city)`: 도시별 파티 손님 수 (가짜)
- `chairs_per_table()`: 테이블 하나에 의자 몇 개 (가짜)
- 문제: "두 도시 손님을 합치면 테이블이 몇 개 필요해?" → **도구 결과로 계산**해야 하는 문제

In [ ]:
from smolagents import CodeAgent, tool


@tool
def get_guest_count(city: str) -> int:
    """Returns how many guests are coming from a city.

    Args:
        city: One of "Gotham" or "Metropolis".
    """
    return {"Gotham": 37, "Metropolis": 26}.get(city, 0)


@tool
def chairs_per_table() -> int:
    """Returns how many chairs fit around one table."""
    return 8


task = (
    "Guests are coming from Gotham and Metropolis. "
    "How many tables do we need so every guest has a chair? Use the tools."
)

results = {}
for AgentClass in [CodeAgent, ToolCallingAgent]:
    print("\n" + "#" * 30, AgentClass.__name__, "#" * 30)
    a = AgentClass(tools=[get_guest_count, chairs_per_table], model=model, max_steps=6)
    out = a.run(task)
    steps = [s for s in a.memory.steps if isinstance(s, ActionStep)]
    results[AgentClass.__name__] = (out, len(steps))

print("\n===== 비교 =====")
print("정답: 37 + 26 = 63명 → 63 ÷ 8 = 7.875 → 올림해서 8개")
for name, (out, n) in results.items():
    print(f"{name:17s} → 답: {out} | 바퀴 수: {n}")

✅ **확인할 것** (작은 모델이라 매번 조금씩 다를 수 있어요)
- **CodeAgent**: 코드 한 덩어리에 `get_guest_count("Gotham") + get_guest_count("Metropolis")`, `math.ceil(...)` 같은 계산까지 넣어 **바퀴가 적게** 끝나는 경우가 많음 → 계산은 **파이썬이** 함
- **ToolCallingAgent**: 도구를 부를 수만 있고 계산 도구는 없음 → 도구 결과(37, 26, 8)를 관찰로 받은 뒤 **모델이 머릿속으로** 더하고 나눠서 `final_answer`에 적음 → 바퀴가 더 많고, 계산 실수 가능성도 있음
- 두 에이전트 모두 답이 `8`이면 성공. 틀린 쪽이 있으면 그 에이전트의 로그를 보고 어디서 틀렸는지 같이 봐요

이게 원문의 "`ToolCallingAgent`는 **변수 처리나 복잡한 도구 호출이 필요 없는 단순한 시스템**에 효과적"이라는 말의 실제 모습이에요.

## 정리

| | CodeAgent | ToolCallingAgent |
|---|---|---|
| 행동 모양 | 파이썬 코드 | JSON (도구 이름 + 인자) |
| 로그 | `Executing parsed code:` | `Calling tool: '...' with arguments: {...}` |
| 모델 답에서 꺼내는 법 | 글 속 코드 블록을 **파싱** | 모델 API의 **`tool_calls` 칸**을 그대로 읽음 |
| 변수·`for`·`if`·계산 | 가능 | 불가 (계산은 모델 머리로) |
| 최종 답 | 코드에서 `final_answer(...)` 호출 | `final_answer` **도구 호출** |
| 잘 맞는 일 | 여러 단계를 엮는 일 | 도구 하나씩 부르면 끝나는 단순한 일 |